# 02 — EDA Univariate


In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
for candidate in [REPO, REPO.parent, REPO.parent.parent]:
    if (candidate / "src" / "neuro").exists():
        REPO = candidate
        break
sys.path.insert(0, str(REPO / "src"))
import os
os.chdir(REPO)
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:

import numpy as np
import pandas as pd
import nibabel as nib
from sklearn.preprocessing import StandardScaler
from neuro.bids import validate_bids, load_participants
from neuro.features import parse_events
from neuro.mlflow_utils import start_run
from neuro import viz

with start_run("02_eda_univariate"):
    report = validate_bids()
    runs = report["runs"]
    participants = load_participants()
    available = runs[runs["bold_exists"]]
    mlflow.log_param("n_runs", len(available))
    print(f"Available runs: {len(available)}")


In [ ]:

t1_path = available[available["t1_path"].notna()].iloc[0]["t1_path"]
t1_data = nib.load(t1_path).get_fdata()
t1_vals = t1_data[t1_data > 0].ravel()
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(t1_vals, bins=80, kde=True, color="#8172B3", ax=ax)
ax.set_title("T1w voxel intensities (non-zero)")
ax.set_xlabel("Intensity")
plt.show()


In [ ]:

row = available.iloc[0]
bold = nib.load(row["bold_path"])
data = bold.get_fdata()
mean_ts = data.reshape(-1, data.shape[-1]).mean(axis=0)
ts_df = pd.DataFrame({"volume": np.arange(len(mean_ts)), "signal": mean_ts})
fig, ax = plt.subplots(figsize=(10, 3))
sns.lineplot(data=ts_df, x="volume", y="signal", ax=ax, color="#4C72B0")
ax.set_title(f"Global mean BOLD — {row['subject']}")
plt.show()


In [ ]:

events = pd.read_csv(row["events_path"], sep="\t")
viz.plot_trial_counts(events)
plt.show()
demo = participants.groupby(["group_short", "sex"]).size().reset_index(name="n")
sns.barplot(data=demo, x="group_short", y="n", hue="sex", palette="Set2")
plt.title("Participants by group and sex")
plt.show()
display(demo)
